In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import LineString
from shapely.ops import unary_union
import networkx as nx
import branca.colormap as cm

In [ ]:
# Analyse de la part d’espace par usage (trottoirs, pistes cyclables, bus, chaussée)
# à partir du réseau routier linéaire et des objets surfaciques du cadastre routier (SITG)

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, box
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
import warnings

# Filtrer certains warnings gênants
warnings.filterwarnings("ignore", category=UserWarning, module="pyproj")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="pyogrio")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*unary_union.*")

# -------------------------------------------------
# 1. Chargement et préparation des données
# -------------------------------------------------
print("Chargement des shapefiles…")
lines = gpd.read_file("../Data/AGGLO_HIERARCHIE_ROUTE-SHP/AGGLO_HIERARCHIE_ROUTE.shp").to_crs("EPSG:2056")
surfaces = gpd.read_file("../Data/CAD_DOMROUTIER_OBJETS_NIV0-SHP/CAD_DOMROUTIER_OBJETS_NIV0.shp").to_crs("EPSG:2056")

# Nettoyage des géométries surfaciques
surfaces = surfaces[surfaces.is_valid & surfaces.geometry.notnull()].copy()
surfaces = surfaces[surfaces.geometry.area > 1].copy()

# -------------------------------------------------
# 2. Buffer de 100 m autour des lignes
# -------------------------------------------------
print("Création des zones tampons autour des lignes…")
lines = lines.reset_index().rename(columns={"index": "line_index"})
lines["buffer"] = lines.geometry.buffer(100, cap_style="round")
lines_buffered = lines.set_geometry("buffer")

# -------------------------------------------------
# 3. Intersections surfaciques / linéaires
# -------------------------------------------------
print("Calcul de l’intersection entre surfaces et buffers de lignes…")
intersected = gpd.overlay(surfaces, lines_buffered, how="intersection")
intersected["area"] = intersected.geometry.area

# -------------------------------------------------
# 4. Agrégation des surfaces par ligne et usage
# -------------------------------------------------
print("Agrégation des surfaces par tronçon…")
intersected = intersected[intersected["area"] > 0].copy()
grouped = intersected.groupby(["line_index", "OBJET"])["area"].sum().unstack(fill_value=0).reset_index()

# Totaux + ratios
grouped["total"] = grouped.drop(columns=["line_index"]).sum(axis=1)
if "t" in grouped.columns:
    grouped["ratio_trottoir"] = grouped["t"] / grouped["total"]
if "pc" in grouped.columns:
    grouped["ratio_cyclable"] = grouped["pc"] / grouped["total"]
if "tc" in grouped.columns:
    grouped["ratio_tc"] = grouped["tc"] / grouped["total"]
if "c" in grouped.columns:
    grouped["ratio_chaussée"] = grouped["c"] / grouped["total"]

# -------------------------------------------------
# 5. Fusion avec les lignes originales
# -------------------------------------------------
print("Fusion avec les données linéaires originales…")
lines = lines.merge(grouped, on="line_index", how="left")
lines = lines.set_geometry("geometry").drop(columns="buffer")






In [ ]:
import matplotlib.pyplot as plt

mask = (grouped["ratio_trottoir"] > 0) & (grouped["ratio_cyclable"] > 0)
grouped.loc[mask, "ratio_trottoir"].hist(bins=30, alpha=0.6, label="Trottoir")
grouped.loc[mask, "ratio_cyclable"].hist(bins=30, alpha=0.6, label="Cyclable")
plt.xlabel("Ratio")
plt.ylabel("Nombre de tronçons")
plt.title("Distribution des ratios trottoir et cyclable (coexistence)")
plt.legend()
plt.show()

In [ ]:
# -------------------------------------------------
# 6. Définir les faisceaux d'intérêt sous forme de cercles le long des lignes
# -------------------------------------------------
print("Définition des faisceaux d’analyse…")
faisceaux = [
    LineString([(6.020712, 46.243278), (6.1422, 46.2102)]),
    LineString([(6.081205, 46.143815), (6.102600, 46.162638), (6.1397, 46.1811)])
]
gdf_faisceaux = gpd.GeoDataFrame(geometry=faisceaux, crs="EPSG:4326").to_crs("EPSG:2056")

# Créer des cercles tous les 1km avec un rayon de 650m, puis fusionner
circles = []
for line in gdf_faisceaux.geometry:
    for dist in range(0, int(line.length), 1400):
        pt = line.interpolate(dist)
        circles.append(pt.buffer(750))

faisceau_union = gpd.GeoSeries(circles, crs="EPSG:2056").unary_union
area_gdf = gpd.GeoDataFrame(geometry=[faisceau_union], crs="EPSG:2056")
area_union = faisceau_union

# -------------------------------------------------
# 7. Visualisation interactive
# -------------------------------------------------
print("Création de la carte interactive…")
lines_wgs = lines.to_crs("EPSG:4326")
faisceaux_wgs = area_gdf.to_crs("EPSG:4326")
center = lines_wgs.geometry.unary_union.centroid


m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron",control_scale = True, metric=True)


# Palettes avec quantiles
def quantile_colormap(series, n=6, palette=cm.linear.Greens_09):
    qmin, qmax = series.quantile(0.05), series.quantile(0.95)
    return palette.scale(qmin, qmax)

# Génération des palettes
velo_cmap = quantile_colormap(lines["ratio_cyclable"].dropna(), palette=cm.linear.Greens_09)
trottoir_cmap = quantile_colormap(lines["ratio_trottoir"].dropna(), palette=cm.linear.OrRd_09)

velo_cmap.caption = "Proportion de piste cyclable (échelle quantile)"
trottoir_cmap.caption = "Proportion de trottoir (échelle quantile)"
velo_cmap.add_to(m)
trottoir_cmap.add_to(m)

def style_ratio(field, cmap):
    def _style(feature):
        val = feature["properties"].get(field, 0)
        try:
            return {"color": cmap(float(val) or 0), "weight": 2, "opacity": 1}
        except:
            return {"color": "#cccccc", "weight": 1, "opacity": 0.5}
    return _style

# Couche trottoirs (sur tout le territoire)
folium.GeoJson(
    lines_wgs,
    name="Ratio trottoir",
    style_function=style_ratio("ratio_trottoir", trottoir_cmap),
    tooltip=folium.GeoJsonTooltip(fields=["ratio_trottoir"], aliases=["Ratio trottoir"])
).add_to(m)

# Couche pistes cyclables (sur tout le territoire)
folium.GeoJson(
    lines_wgs,
    name="Ratio cyclable",
    style_function=style_ratio("ratio_cyclable", velo_cmap),
    tooltip=folium.GeoJsonTooltip(fields=["ratio_cyclable"], aliases=["Ratio cyclable"])
).add_to(m)

# Couche chaussée (sur tout le territoire) - lignes fines noires
folium.GeoJson(
    lines_wgs,
    style_function=lambda f: {"color": "black", "weight": 0.5, "opacity": 0.8},
    name="Chaussée"
).add_to(m)

# Voile blanc en dehors des faisceaux
print("Ajout du masque en dehors des faisceaux…")
full_bounds = lines_wgs.total_bounds
full_rect = box(*full_bounds)
full_gdf = gpd.GeoDataFrame(geometry=[full_rect], crs="EPSG:4326")
mask_geom = full_gdf.overlay(
    gpd.GeoDataFrame(geometry=[faisceaux_wgs.unary_union], crs=full_gdf.crs),
    how="difference"
)

folium.GeoJson(
    mask_geom,
    name="Voile extérieur",
    style_function=lambda f: {"fillColor": "white", "fillOpacity": 0.85, "weight": 0}
).add_to(m)

# Affichage des zones faisceaux
folium.GeoJson(
    faisceaux_wgs,
    name="Zones d’analyse",
    style_function=lambda f: {"color": "#999999", "weight": 1, "fillOpacity": 0}
).add_to(m)

folium.LayerControl().add_to(m)
m


In [ ]:
import geopandas as gpd
from shapely.ops import nearest_points

lines = gpd.read_file("../Data/AGGLO_HIERARCHIE_ROUTE-SHP/AGGLO_HIERARCHIE_ROUTE.shp").to_crs("EPSG:2056")
surfaces = gpd.read_file("../Data/CAD_DOMROUTIER_OBJETS_NIV0-SHP/CAD_DOMROUTIER_OBJETS_NIV0.shp").to_crs("EPSG:2056")

# 1. Supprimer les géométries invalides (vides ou non valides)
surfaces = surfaces[surfaces.is_valid & surfaces.geometry.notnull()].copy()

# 2. Écarter les polygones trop petits (optionnel, seuil à ajuster)
surfaces = surfaces[surfaces.geometry.area > 1].copy()

from scipy.spatial import cKDTree
import numpy as np


surfaces["centroid"] = surfaces.geometry.centroid

# Coords centroïdes
surface_coords = np.array(list(surfaces["centroid"].apply(lambda geom: (geom.x, geom.y))))

# Coords milieux de lignes
line_midpoints = lines.geometry.interpolate(lines.geometry.length / 2)
line_coords = np.array(list(line_midpoints.apply(lambda geom: (geom.x, geom.y))))

# Index spatial avec cKDTree
tree = cKDTree(line_coords)
distances, indices = tree.query(surface_coords, distance_upper_bound=15)  # 30 m max

# Affectation
surfaces["line_index"] = indices
surfaces["dist"] = distances

# Option : garder que les liens plausibles
surfaces = surfaces[surfaces["dist"] < 15]


surfaces = surfaces[surfaces["dist"] < 15].copy()

surfaces["area"] = surfaces.geometry.area
grouped = surfaces.groupby(["line_index", "OBJET"])["area"].sum().unstack(fill_value=0).reset_index()

# Totaux + ratios
grouped["total"] = grouped.drop(columns="line_index").sum(axis=1)
if "t" in grouped.columns:  # trottoirs
    grouped["ratio_trottoir"] = grouped["t"] / grouped["total"]
if "c" in grouped.columns:  # cyclable
    grouped["ratio_cyclable"] = grouped["pc"] / grouped["total"]


lines = lines.reset_index().rename(columns={"index": "line_index"})
lines = lines.merge(grouped, on="line_index", how="left")


In [ ]:
import folium

# Conversion des géométries
surfaces_wgs = surfaces.to_crs("EPSG:4326")

# Filtrage
trottoirs = surfaces_wgs[surfaces_wgs["OBJET"] == "t"]
cyclables = surfaces_wgs[surfaces_wgs["OBJET"] == "pc"]
transports = surfaces_wgs[surfaces_wgs["OBJET"] == "tc"]
chaussee    = surfaces_wgs[surfaces_wgs["OBJET"] == "ch"]

# Centrage
center = surfaces_wgs.unary_union.centroid

# Nettoyage
cols_keep = ["geometry", "OBJET", "area", "dist"]
layers = {
    "Trottoirs": (trottoirs[cols_keep], "#8da0cb"),
    "Pistes cyclables": (cyclables[cols_keep], "#66c2a5"),
    "Transport public": (transports[cols_keep], "#e9c46a"),
    "Chaussée": (chaussee[cols_keep], "#b0b0b0"),
}

# Création de la carte
m = folium.Map(location=[center.y, center.x], zoom_start=15, tiles="cartodbpositron")

# Ajout des couches
for name, (gdf, color) in layers.items():
    folium.GeoJson(
        gdf,
        name=name,
        style_function=lambda feature, col=color: {
            "fillColor": col,
            "color": col,
            "weight": 0.5,
            "fillOpacity": 0.6,
        },
        tooltip=folium.GeoJsonTooltip(fields=["OBJET", "area", "dist"],
                                      aliases=["Objet", "Surface (m²)", "Distance à la ligne"])
    ).add_to(m)

folium.LayerControl().add_to(m)
m


In [ ]:
lines_wgs = lines.to_crs("EPSG:4326")
import folium
import branca.colormap as cm

# Centrer la carte
center = lines_wgs.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=14, tiles="cartodbpositron")

colormap = cm.linear.Blues_09.scale(0, 1)
colormap.caption = "Proportion de trottoirs"
colormap.add_to(m)

def style_trottoir(feature):
    ratio = feature["properties"].get("ratio_trottoir", 0)
    try:
        ratio = float(ratio)
    except:
        ratio = 0
    return {
        "color": colormap(ratio),
        "weight": 2
    }

folium.GeoJson(
    lines_wgs,
    name="Trottoirs",
    style_function=style_trottoir,
    tooltip=folium.GeoJsonTooltip(fields=["NOM", "CLASSE", "ratio_trottoir"],
                                   aliases=["Nom", "Classe", "Ratio trottoir"],
                                   localize=True)
).add_to(m)

colormap_cyc = cm.linear.YlGnBu_09.scale(0, 1)
colormap_cyc.caption = "Proportion de pistes cyclables"
colormap_cyc.add_to(m)

def style_cyclable(feature):
    ratio = feature["properties"].get("ratio_cyclable", 0)
    try:
        ratio = float(ratio)
    except:
        ratio = 0
    return {
        "color": colormap_cyc(ratio),
        "weight": 2
    }

folium.GeoJson(
    lines_wgs,
    name="Cyclabilité",
    style_function=style_cyclable,
    tooltip=folium.GeoJsonTooltip(fields=["NOM", "CLASSE", "ratio_cyclable"],
                                   aliases=["Nom", "Classe", "Ratio cyclable"],
                                   localize=True)
).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:
print(lines_wgs.columns[lines_wgs.columns.duplicated()])


In [ ]:
lines_wgs